In [2]:
import joblib
import numpy as np
import pandas as pd
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
import openai
import warnings
warnings.filterwarnings('ignore')


In [3]:
# ============================================================================
# STEP 1: LOAD ENVIRONMENT VARIABLES (SECURE API KEY HANDLING)
# ============================================================================
print("=" * 60)
print("STEP 1: LOADING ENVIRONMENT VARIABLES")
print("=" * 60)

# Load environment variables from .env file
load_dotenv()

# Check if OpenAI API key is available
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY and OPENAI_API_KEY != "sk-your-actual-api-key-here":
    openai.api_key = OPENAI_API_KEY
    USE_OPENAI = True
    print("✅ OpenAI API key loaded successfully")
    print("   Using OpenAI for dynamic retention strategies")
else:
    USE_OPENAI = False
    print("⚠️ No valid OpenAI API key found.")
    print("   Using template-based strategies (no API key required)")
    print("\n   To enable OpenAI strategies:")
    print("   1. Get an API key from: https://platform.openai.com/api-keys")
    print("   2. Create a .env file in the project root")
    print("   3. Add: OPENAI_API_KEY=sk-your-actual-key-here")
    print("   4. Install: pip install python-dotenv openai")


STEP 1: LOADING ENVIRONMENT VARIABLES
✅ OpenAI API key loaded successfully
   Using OpenAI for dynamic retention strategies


In [4]:
# ============================================================================
# STEP 2: LOAD MODEL AND DATA
# ============================================================================
print("\n" + "=" * 60)
print("STEP 2: LOADING MODEL AND DATA")
print("=" * 60)

try:
    model = keras.models.load_model("../models/best_ann_model.keras")
    print("✅ Model loaded successfully")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("   Please run 02_ann_model.ipynb first")

try:
    X_test = joblib.load("../models/X_test_processed.pkl")
    y_test = joblib.load("../models/y_test_processed.pkl")
    optimal_threshold = joblib.load("../models/optimal_threshold.pkl")
    print("✅ Test data loaded successfully")
except Exception as e:
    print(f"❌ Error loading test data: {e}")

# Load feature importance if available
try:
    feature_importance = pd.read_csv("../models/feature_importance.csv")
    print("✅ Feature importance loaded")
except:
    feature_importance = None
    print("⚠️ No feature importance file found")

# Load original data for customer information
try:
    df_original = pd.read_csv("../data/processed/processed_churn_data.csv")
    print(f"✅ Original data loaded: {df_original.shape}")
except Exception as e:
    print(f"❌ Error loading original data: {e}")



STEP 2: LOADING MODEL AND DATA
✅ Model loaded successfully
✅ Test data loaded successfully
✅ Feature importance loaded
✅ Original data loaded: (10000, 18)


In [5]:
# ============================================================================
# STEP 3: PREPARE CUSTOMER DATA FOR PREDICTIONS
# ============================================================================
print("\n" + "=" * 60)
print("STEP 3: PREPARING CUSTOMER DATA")
print("=" * 60)

# Get the test customers (last 20% of the dataset)
test_size = len(y_test)
X_test_original = df_original.iloc[-test_size:].reset_index(drop=True)
print(f"✅ Test customers: {len(X_test_original)}")

# Convert to numpy arrays for prediction
if isinstance(X_test, pd.DataFrame):
    X_test = X_test.values
if isinstance(y_test, pd.Series):
    y_test = y_test.values


STEP 3: PREPARING CUSTOMER DATA
✅ Test customers: 2000


In [6]:
# ============================================================================
# STEP 4: GENERATE PREDICTIONS
# ============================================================================
print("\n" + "=" * 60)
print("STEP 4: GENERATING PREDICTIONS")
print("=" * 60)

y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > optimal_threshold).astype(int)

print(f"✅ Predictions generated for {len(y_pred_prob)} customers")



STEP 4: GENERATING PREDICTIONS
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
✅ Predictions generated for 2000 customers


In [7]:
# ============================================================================
# STEP 5: DEFINE RISK LEVEL FUNCTION
# ============================================================================
print("\n" + "=" * 60)
print("STEP 5: DEFINING RISK LEVELS")
print("=" * 60)

def get_risk_level(probability):
    """Categorize customers by risk level based on churn probability"""
    if probability < 0.3:
        return "Low Risk"
    elif probability < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

print("✅ Risk levels defined:")
print("   - Low Risk: < 30%")
print("   - Medium Risk: 30-70%")
print("   - High Risk: > 70%")


STEP 5: DEFINING RISK LEVELS
✅ Risk levels defined:
   - Low Risk: < 30%
   - Medium Risk: 30-70%
   - High Risk: > 70%


In [8]:
# ============================================================================
# STEP 6: CREATE CUSTOMER PROFILES
# ============================================================================
print("\n" + "=" * 60)
print("STEP 6: CREATING CUSTOMER PROFILES")
print("=" * 60)

customer_profiles = []
num_customers = min(50, len(X_test))  # Generate for 50 customers

for i in range(num_customers):
    profile = {
        'customer_id': i + 1001,
        'age': X_test_original.iloc[i]['Age'] if 'Age' in X_test_original.columns else 'Unknown',
        'balance': X_test_original.iloc[i]['Balance'] if 'Balance' in X_test_original.columns else 0,
        'products': X_test_original.iloc[i]['NumOfProducts'] if 'NumOfProducts' in X_test_original.columns else 1,
        'active': bool(X_test_original.iloc[i]['IsActiveMember']) if 'IsActiveMember' in X_test_original.columns else False,
        'credit_score': X_test_original.iloc[i]['CreditScore'] if 'CreditScore' in X_test_original.columns else 600,
        'geography': X_test_original.iloc[i]['Geography'] if 'Geography' in X_test_original.columns else 'Unknown',
        'gender': X_test_original.iloc[i]['Gender'] if 'Gender' in X_test_original.columns else 'Unknown',
        'churn_probability': y_pred_prob[i],
        'predicted_churn': 'Yes' if y_pred[i] == 1 else 'No',
        'actual_churn': 'Yes' if y_test[i] == 1 else 'No',
        'risk_level': get_risk_level(y_pred_prob[i])
    }
    customer_profiles.append(profile)

print(f"✅ Created profiles for {len(customer_profiles)} customers")



STEP 6: CREATING CUSTOMER PROFILES
✅ Created profiles for 50 customers


In [9]:
# ============================================================================
# STEP 7: TEMPLATE-BASED RETENTION STRATEGIES (FALLBACK)
# ============================================================================
print("\n" + "=" * 60)
print("STEP 7: DEFINING TEMPLATE-BASED STRATEGIES")
print("=" * 60)

def get_template_strategy(customer):
    """Generate personalized retention strategy using templates (no API key needed)"""
    
    strategy = f"🎯 CUSTOMER #{customer['customer_id']} - {customer['risk_level']}\n"
    strategy += f"📊 Customer Profile: Age {customer['age']}, {customer['products']} product(s), "
    strategy += f"Balance: ${customer['balance']:,.2f}\n"
    strategy += f"⚠️ Churn Risk: {customer['churn_probability']:.1%}\n\n"
    strategy += "💡 RECOMMENDED RETENTION ACTIONS:\n"
    
    # High Risk customers
    if customer['risk_level'] == "High Risk":
        strategy += "\n🔴 URGENT INTERVENTION REQUIRED:\n"
        
        if customer['products'] <= 1:
            strategy += "1️⃣ Cross-sell additional products:\n"
            strategy += "   - Offer bundled product package with 20% discount for 6 months\n"
            strategy += "   - Schedule personal banker consultation within 48 hours\n"
        
        if customer['balance'] > 50000:
            strategy += "2️⃣ Premium retention offer:\n"
            strategy += "   - Upgrade to premium banking with first year free\n"
            strategy += "   - Assign dedicated relationship manager\n"
            strategy += "   - Offer exclusive investment consultation\n"
        else:
            strategy += "2️⃣ Financial incentive:\n"
            strategy += "   - Cashback offer on next 10 transactions (up to $100)\n"
            strategy += "   - Waive monthly fees for 6 months\n"
        
        if not customer['active']:
            strategy += "3️⃣ Re-engagement campaign:\n"
            strategy += "   - Send personalized re-activation email with bonus offer\n"
            strategy += "   - Offer $50 welcome back bonus after first transaction\n"
        
        strategy += "\n📞 Immediate action: Priority phone call within 24 hours"
    
    # Medium Risk customers
    elif customer['risk_level'] == "Medium Risk":
        strategy += "\n🟠 PROACTIVE RETENTION CAMPAIGN:\n"
        
        if not customer['active']:
            strategy += "1️⃣ Re-engagement offer:\n"
            strategy += "   - 2% cashback on next 5 transactions\n"
            strategy += "   - Free credit score monitoring for 3 months\n"
        else:
            strategy += "1️⃣ Loyalty reward:\n"
            strategy += "   - Double loyalty points for 3 months\n"
            strategy += "   - Referral bonus program ($50 per referral)\n"
        
        strategy += "2️⃣ Value-added services:\n"
        strategy += "   - Free financial planning webinar invitation\n"
        strategy += "   - Pre-approved credit line increase\n"
        
        strategy += "\n📧 Follow-up: Email campaign with personalized offers"
    
    # Low Risk customers
    else:
        strategy += "\n🟢 LOYALTY REINFORCEMENT:\n"
        
        strategy += "1️⃣ Appreciation reward:\n"
        strategy += "   - $10 coffee voucher or gift card\n"
        strategy += "   - Birthday bonus ($5 cashback)\n"
        
        strategy += "2️⃣ Engagement opportunity:\n"
        strategy += "   - Early access to new banking features\n"
        strategy += "   - Invitation to customer appreciation event\n"
        
        strategy += "\n💌 Keep them happy: Monthly newsletter with personalized tips"
    
    return strategy

print("✅ Template strategies defined")


STEP 7: DEFINING TEMPLATE-BASED STRATEGIES
✅ Template strategies defined


In [10]:
# ============================================================================
# STEP 8: OPENAI-BASED STRATEGY GENERATOR (IF API KEY AVAILABLE)
# ============================================================================
print("\n" + "=" * 60)
print("STEP 8: DEFINING OPENAI STRATEGY GENERATOR")
print("=" * 60)

def get_openai_strategy(customer):
    """Generate personalized strategy using OpenAI (requires API key)"""
    
    prompt = f"""Generate a personalized customer retention strategy for a bank customer with the following profile:

CUSTOMER PROFILE:
- Risk Level: {customer['risk_level']}
- Age: {customer['age']}
- Number of Products: {customer['products']}
- Active Member: {'Yes' if customer['active'] else 'No'}
- Account Balance: ${customer['balance']:,.2f}
- Churn Probability: {customer['churn_probability']:.1%}

Please provide:
1. A brief analysis of why this customer might be at risk (2-3 sentences)
2. Three specific, actionable retention offers tailored to this customer's profile
3. The expected impact of each offer

Format the response in a clear, professional manner with bullet points."""
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are an expert in customer retention and banking services. Provide practical, personalized retention strategies."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"⚠️ OpenAI API error: {e}")
        return get_template_strategy(customer)  # Fallback to template

if USE_OPENAI:
    print("✅ OpenAI strategy generator ready")
else:
    print("⚠️ OpenAI not available - using templates only")


STEP 8: DEFINING OPENAI STRATEGY GENERATOR
✅ OpenAI strategy generator ready


In [11]:
# ============================================================================
# STEP 9: GENERATE STRATEGIES FOR ALL CUSTOMERS
# ============================================================================
print("\n" + "=" * 80)
print("STEP 9: GENERATING CUSTOMER RETENTION STRATEGIES")
print("=" * 80)

strategies = []
risk_counts = {'Low Risk': 0, 'Medium Risk': 0, 'High Risk': 0}

for i, customer in enumerate(customer_profiles):
    risk_counts[customer['risk_level']] += 1
    
    # Use OpenAI if available, otherwise use template
    if USE_OPENAI:
        strategy_text = get_openai_strategy(customer)
    else:
        strategy_text = get_template_strategy(customer)
    
    strategies.append({
        'customer_id': customer['customer_id'],
        'risk_level': customer['risk_level'],
        'churn_probability': f"{customer['churn_probability']:.1%}",
        'predicted_churn': customer['predicted_churn'],
        'actual_churn': customer['actual_churn'],
        'age': customer['age'],
        'products': customer['products'],
        'balance': f"${customer['balance']:,.2f}",
        'active': customer['active'],
        'strategy': strategy_text
    })
    
    # Print progress every 10 customers
    if (i + 1) % 10 == 0:
        print(f"✅ Generated strategies for {i + 1} customers")



STEP 9: GENERATING CUSTOMER RETENTION STRATEGIES
⚠️ OpenAI API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
⚠️ OpenAI API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
⚠️ OpenAI API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
⚠️ OpenAI API error: You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.
⚠️ OpenAI API error: You exceeded your current quota, please check your plan and billing details. 